# Onset: турновая классификация

Задача — предсказать `t*`, индекс первой реплики, где обман становится текстуально
наблюдаем. Определён ТОЛЬКО для скамов. gold-dev: 64 скама из 100, gold-test:
191 из 303.

**Архитектура.** `xlm-roberta-base` + голова токенной классификации (2 класса).
Диалог кодируется целиком, метка ставится на первый токен маркера каждой реплики
(`[i|говорящий]`), остальные позиции — `-100`. На инференсе берётся вероятность
класса «здесь онсет» в позициях маркеров, `t̂* = argmax` по репликам.

**Обучение** — только на скам-диалогах train (696, silver-онсеты), валидация на
скамах val (103, silver).

**Отбор чекпоинта** — по **OAD = среднее |t̂* − t*|** на gold-dev, МИНИМУМ.

⚠ Согласие silver с gold по онсету: Krippendorff α 0.316, exact 0.58, ±1 0.615,
MAE 1.19. Сигнал шумный, результат будет соответствующим.

⚠ 51 диалог имеет корпусный `hard_negative=1` при gold `is_scam=1` (11 в dev,
40 в test). Вопрос об их скамовости НЕ решается: все метрики считаются в двух
вариантах — на всех скамах и с исключением спорных.

Протокол: сиды 20260721/22/23, lr 2e-5, wd 0.01, warmup 0.2, max_length 512,
10 эпох, patience=3, `gold_dev_split.json` с ассертами. Снимки предсказаний на
dev и test на каждой эпохе.

## Данные

In [ ]:
import os, sys, json, csv, glob, pathlib, time, copy
import numpy as np

def find_one(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"не найден {name}"
    return hits[0]

CORPUS, GOLD, GOLD_DEV = find_one("corpus_v1.jsonl"), find_one("gold_test_v1.csv"), find_one("gold_dev_split.json")
OUT = pathlib.Path("/kaggle/working/onset"); OUT.mkdir(parents=True, exist_ok=True)
TB_ROOT = pathlib.Path("/kaggle/working/tb")

MODEL_NAME = "xlm-roberta-base"
SEEDS = [20260721, 20260722, 20260723]
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.2
EPOCHS, MAX_LENGTH, BATCH_SIZE, PATIENCE = 10, 512, 8, 3

import torch, transformers
DEVICE, USE_FP16 = ("cuda", True) if torch.cuda.is_available() else ("cpu", False)
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("устройство:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "",
      "| GPU:", torch.cuda.device_count() if DEVICE == "cuda" else 0)

def read_corpus(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line: out.append(json.loads(line))
    return out

def turn_prefix(t):
    return f"[{t['i']}|{t['speaker']}] "

def build_text_and_starts(rec):
    """Текст диалога + символьные позиции начала каждой реплики."""
    parts, starts, pos = [], [], 0
    for t in rec["turns"]:
        pre = turn_prefix(t)
        starts.append(pos)
        chunk = pre + t["text"]
        parts.append(chunk)
        pos += len(chunk) + 1          # +1 на "\n"
    return "\n".join(parts), starts

records = read_corpus(CORPUS)
gold = {}
with open(GOLD, encoding="utf-8-sig", newline="") as f:
    for r in csv.DictReader(f):
        o = (r.get("onset") or "").strip()
        gold[r["dialog_id"].strip()] = {"is_scam": int(r["is_scam"]),
                                        "onset": int(o) if o else None}

train_recs = [r for r in records if r["split"] == "train" and r["is_scam"] == 1 and r.get("onset")]
val_recs   = [r for r in records if r["split"] == "val"   and r["is_scam"] == 1 and r.get("onset")]
for r in train_recs + val_recs:
    r["t_star"] = int(r["onset"])          # silver

by_id = {r["id"]: r for r in records}
sm = json.load(open(GOLD_DEV, encoding="utf-8")); assign = sm["assignment"]
dev_recs, test_recs = [], []
for did, part in assign.items():
    g = gold[did]
    if g["is_scam"] != 1:
        continue                            # онсет определён только для скамов
    src = by_id[did]
    r = {k: v for k, v in src.items() if k not in ("is_scam", "onset")}
    r["t_star"] = int(g["onset"])           # gold
    r["t_star_silver"] = int(src["onset"]) if src.get("onset") else None
    r["disputed"] = int(src.get("hard_negative", 0) == 1)   # спорный: hard_neg=1 при gold scam=1
    (dev_recs if part == "dev" else test_recs).append(r)

gd = {r.get("scenario_group") for r in dev_recs}
gt = {r.get("scenario_group") for r in test_recs}
assert not (gd & gt), "группы scenario_group разорваны между dev и test"
assert len({len(dev_recs), len(val_recs), len(test_recs)}) == 3, "размеры наборов совпали"
print(f"\nразбиение: seed={sm['seed']}, всего dev {sm['n_dev']} / test {sm['n_test']}")
print(f"скамы: train {len(train_recs)} (silver) / val {len(val_recs)} (silver) / "
      f"gold-dev {len(dev_recs)} / gold-test {len(test_recs)}")
print(f"спорных (hard_negative=1 при gold scam=1): dev {sum(r['disputed'] for r in dev_recs)}, "
      f"test {sum(r['disputed'] for r in test_recs)}")

## Токенизация: метки на маркерах реплик

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(rec):
    """Диалог → input_ids + labels, где на первом токене маркера реплики стоит
    0/1 (1 = здесь онсет), а везде ещё -100. Реплики, не влезшие в окно, теряются."""
    text, starts = build_text_and_starts(rec)
    enc = tok(text, truncation=True, max_length=MAX_LENGTH, padding="max_length",
              return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")
    labels = [-100] * len(enc["input_ids"])
    turn_pos = []
    for ti, ch in enumerate(starts, start=1):
        idx = None
        for k, (a, b) in enumerate(offsets):
            if a == b:                       # спецтокен / паддинг
                continue
            if a <= ch < b or a == ch:
                idx = k; break
        if idx is None:
            continue                          # реплика за пределами окна
        labels[idx] = 1 if ti == rec["t_star"] else 0
        turn_pos.append((ti, idx))
    enc["labels"] = labels
    return enc, turn_pos

class OnsetDataset(torch.utils.data.Dataset):
    def __init__(self, recs):
        self.items, self.meta = [], []
        for r in recs:
            enc, tp = encode(r)
            self.items.append(enc)
            self.meta.append({"id": r["id"], "lang": r.get("lang"), "t_star": r["t_star"],
                              "n_turns": len(r["turns"]), "turns_in_window": len(tp),
                              "t_star_in_window": any(ti == r["t_star"] for ti, _ in tp),
                              "disputed": r.get("disputed", 0),
                              "t_star_silver": r.get("t_star_silver")})
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        return {k: torch.tensor(v) for k, v in self.items[i].items()}

ds_train, ds_val = OnsetDataset(train_recs), OnsetDataset(val_recs)
ds_dev,  ds_test = OnsetDataset(dev_recs),  OnsetDataset(test_recs)
for nm, ds in (("train", ds_train), ("val", ds_val), ("gold-dev", ds_dev), ("gold-test", ds_test)):
    lost = sum(1 for m in ds.meta if not m["t_star_in_window"])
    print(f"  {nm:9s}: n={len(ds)}, диалогов с t* за пределами 512 токенов: {lost}")

## Обучение

Отбор чекпоинта — по минимуму OAD на gold-dev.

In [ ]:
from transformers import (AutoModelForTokenClassification, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)

def decode_batch(logits, labels):
    """Логиты токенов → t̂* и t* по диалогам. Позиции реплик = там, где label != -100."""
    p_onset = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, :, 1]
    pred, true, probs = [], [], []
    for i in range(len(labels)):
        pos = np.where(np.asarray(labels[i]) != -100)[0]
        if len(pos) == 0:
            pred.append(1); true.append(1); probs.append([]); continue
        pr = p_onset[i][pos]
        pred.append(int(np.argmax(pr)) + 1)          # индексы реплик с 1
        lab = np.asarray(labels[i])[pos]
        true.append(int(np.argmax(lab)) + 1 if lab.max() == 1 else -1)
        probs.append([float(x) for x in pr])
    return np.array(pred), np.array(true), probs

def onset_metrics(pred, true):
    ok = true > 0
    if ok.sum() == 0:
        return {"oad": float("nan"), "exact": 0.0}
    d = np.abs(pred[ok] - true[ok])
    return {"oad": float(d.mean()), "exact": float((d == 0).mean()),
            "within1": float((d <= 1).mean()), "within2": float((d <= 2).mean()),
            "median_abs": float(np.median(d)),
            "bias": float((pred[ok] - true[ok]).mean())}

def run_seed(seed):
    set_seed(seed); t0 = time.time()
    tb_dir = TB_ROOT / f"onset_seed{seed}"; tb_dir.mkdir(parents=True, exist_ok=True)
    os.environ["TENSORBOARD_LOGGING_DIR"] = str(tb_dir)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=2, torch_dtype=torch.float32)

    dev_snaps, test_snaps = [], []
    capture = {"on": True}

    def compute_metrics(ep):
        logits, labels = ep
        pred, true, probs = decode_batch(logits, labels)
        if capture["on"] and len(labels) in (len(ds_dev), len(ds_test)):
            snap = {"pred": pred.tolist(), "true": true.tolist(), "probs": probs}
            (dev_snaps if len(labels) == len(ds_dev) else test_snaps).append(snap)
        return onset_metrics(pred, true)

        # ФИНАЛЬНАЯ модель: веса обязаны пережить сессию. save_total_limit=2 и
        # обязательный trainer.save_model(...) наружу — см. CLAUDE.md,
        # «Конвенции: сохранение моделей».
    targs = TrainingArguments(
        output_dir=str(OUT / f"ckpt_seed{seed}"),
        learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        seed=seed, data_seed=seed, fp16=USE_FP16,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_gold_dev_oad",
        greater_is_better=False,                 # OAD минимизируем
        save_total_limit=2, logging_steps=10,
        report_to="tensorboard", logging_dir=str(tb_dir),
    )
    trainer = Trainer(model=model, args=targs, train_dataset=ds_train,
                      eval_dataset={"silver_val": ds_val, "gold_dev": ds_dev,
                                    "gold_test": ds_test},
                      processing_class=tok, compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = copy.deepcopy(trainer.state.log_history)

    gv = [h for h in hist if "eval_gold_dev_oad" in h]
    tr = [h for h in hist if "loss" in h and not any(f"eval_{k}_loss" in h
          for k in ("silver_val", "gold_dev", "gold_test"))]
    per_epoch = []
    for h in gv:
        ep = h["epoch"]
        v = [x["loss"] for x in tr if ep - 1 < x["epoch"] <= ep]
        rec = {"epoch": int(round(ep)), "seed": seed,
               "train_loss": float(sum(v) / len(v)) if v else None,
               "gold_dev_oad": float(h["eval_gold_dev_oad"]),
               "gold_dev_exact": float(h["eval_gold_dev_exact"])}
        for src, pfx in (("eval_silver_val", "silver_val"), ("eval_gold_test", "gold_test")):
            g = next((x for x in hist if f"{src}_oad" in x and x["epoch"] == ep), None)
            if g:
                rec[f"{pfx}_oad"] = float(g[f"{src}_oad"])
                rec[f"{pfx}_exact"] = float(g[f"{src}_exact"])
        per_epoch.append(rec)
    best_epoch = min(per_epoch, key=lambda x: x["gold_dev_oad"])["epoch"]

    assert len(dev_snaps) == len(per_epoch) == len(test_snaps), \
        f"эпох {len(per_epoch)}, снимков dev {len(dev_snaps)}, test {len(test_snaps)}"
    for tag, snaps, ds in (("dev", dev_snaps, ds_dev), ("test", test_snaps, ds_test)):
        json.dump({"seed": seed, "meta": ds.meta,
                   "epochs": [p["epoch"] for p in per_epoch], "per_epoch": snaps},
                  open(OUT / f"gold_{tag}_per_epoch_seed{seed}.json", "w", encoding="utf-8"),
                  ensure_ascii=False)
    json.dump(hist, open(OUT / f"history_seed{seed}.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)

    capture["on"] = False
    res = {"seed": seed, "per_epoch": per_epoch, "best_epoch": best_epoch,
           "epochs_ran": len(per_epoch),
           "best_gold_dev_oad": float(trainer.state.best_metric),
           "minutes": round((time.time() - t0) / 60, 1)}
    del trainer, model; torch.cuda.empty_cache()
    print(f"[seed {seed}] {res['minutes']} мин | эпох {res['epochs_ran']} | "
          f"лучшая эпоха {best_epoch} | gold-dev OAD {res['best_gold_dev_oad']:.4f}")
    return res

ALL = []
for s in SEEDS:
    print("=" * 74); print(f"СИД {s}"); print("=" * 74)
    ALL.append(run_seed(s))

## Ход по эпохам и сохранение

In [ ]:
print(f"{'сид':>10} {'эп':>3} {'train':>8} {'val OAD':>8} {'dev OAD':>8} {'dev exact':>10} {'test OAD':>9}")
print("-" * 62)
for r in ALL:
    for p in r["per_epoch"]:
        m = " <" if p["epoch"] == r["best_epoch"] else ""
        print(f"{p['seed']:>10} {p['epoch']:>3} {p['train_loss']:>8.4f} "
              f"{p.get('silver_val_oad', float('nan')):>8.3f} {p['gold_dev_oad']:>8.3f} "
              f"{p['gold_dev_exact']:>10.4f} {p.get('gold_test_oad', float('nan')):>9.3f}{m}")
    print("-" * 62)
print("< лучшая эпоха по МИНИМУМУ OAD на gold-dev")
print("лучшие эпохи:", [r["best_epoch"] for r in ALL],
      "| отработано:", [r["epochs_ran"] for r in ALL], f"из {EPOCHS}")

json.dump({"model": MODEL_NAME, "seeds": SEEDS,
           "config": {"lr": LR, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
                      "epochs": EPOCHS, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
                      "patience": PATIENCE, "selection_metric": "eval_gold_dev_oad (min)",
                      "head": "token classification on turn markers",
                      "device": DEVICE, "n_gpu": torch.cuda.device_count() if DEVICE == "cuda" else 0,
                      "versions": {"torch": torch.__version__,
                                   "transformers": transformers.__version__,
                                   "cuda": torch.version.cuda,
                                   "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}},
           "counts": {"train_scam": len(ds_train), "val_scam": len(ds_val),
                      "dev_scam": len(ds_dev), "test_scam": len(ds_test),
                      "dev_disputed": sum(m["disputed"] for m in ds_dev.meta),
                      "test_disputed": sum(m["disputed"] for m in ds_test.meta),
                      "test_t_star_out_of_window": sum(1 for m in ds_test.meta
                                                       if not m["t_star_in_window"])},
           "runs": ALL, "best_epochs": [r["best_epoch"] for r in ALL],
           "epochs_ran": [r["epochs_ran"] for r in ALL]},
          open(OUT / "results_raw.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("\nсохранено:", sorted(p.name for p in OUT.glob("*.json")))